# Tapi · Cómo reducir las consultas a DeltaPay

Mi objetivo fue entender por qué DeltaPay superaba las 6 consultas por pago por semana y qué consultas podía evitar sin comprometer la detección de facturas. Primero revisé la calidad de los datos, después comparé alternativas y al final usé octubre para probar si el patrón aprendido en agosto y septiembre se repetía.

Los CSV se descargan desde GitHub. Ejecutá las celdas en orden.

In [ ]:
%pip -q install duckdb==1.4.3

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import duckdb

url_base = "https://raw.githubusercontent.com/DaianaSep/tapi_Challenge/main"
nombres = {
    "cuentas": "B_agenda_cuentas.csv",
    "consultas": "B_agenda_consultas.csv",
    "pagos": "B_agenda_pagos.csv",
}

carpeta = Path("data")
carpeta.mkdir(exist_ok=True)

nueva_conexion = duckdb.connect()
try:
    for tabla, archivo in nombres.items():
        destino = carpeta / archivo
        urlretrieve(f"{url_base}/{archivo}", destino)
        nueva_conexion.execute(
            f"CREATE TABLE {tabla} AS "
            "SELECT * FROM read_csv(?, header=true, all_varchar=true)",
            [str(destino)],
        )

except Exception:
    nueva_conexion.close()
    raise

if "conexion" in globals():
    conexion.close()
conexion = nueva_conexion

def consultar(sql):
    return conexion.execute(sql).df()

print("CSV descargados desde GitHub. Tablas cuentas, consultas y pagos listas.")


Cargamos inicialmente todas las columnas como texto para conservar IDs y timestamps originales. Los campos vacíos se leen como NULL. Más adelante convertiremos fechas y montos para los cálculos que lo requieran.

## Preparación y calidad de los datos

Primero conté los registros de cada tabla para dimensionar el análisis. Después verifiqué que cada cuenta, consulta y pago tuviera un identificador informado y único.

In [ ]:
consultar("""
SELECT 'cuentas' AS tabla, COUNT(*) AS filas FROM cuentas
UNION ALL
SELECT 'consultas' AS tabla, COUNT(*) AS filas FROM consultas
UNION ALL
SELECT 'pagos' AS tabla, COUNT(*) AS filas FROM pagos
""")

### Identificadores de cuentas

Comparé filas, IDs informados e IDs distintos para identificar cada cuenta a la que asignar una frecuencia.

In [ ]:
consultar("""
SELECT
    COUNT(*) AS filas,
    COUNT(account_id) AS filas_con_id,
    COUNT(DISTINCT account_id) AS cuentas_distintas
FROM cuentas
""")

### Identificadores de consultas

Comparé filas, IDs informados e IDs distintos para contar intentos sin duplicarlos.

In [ ]:
consultar("""
SELECT
    COUNT(*) AS filas,
    COUNT(consulta_id) AS filas_con_id,
    COUNT(DISTINCT consulta_id) AS consultas_distintas
FROM consultas
""")

### Identificadores de pagos

Comparé filas, IDs informados e IDs distintos para contar pagos sin inflar el denominador.

In [ ]:
consultar("""
SELECT
    COUNT(*) AS filas,
    COUNT(pago_id) AS filas_con_id,
    COUNT(DISTINCT pago_id) AS pagos_distintos
FROM pagos
""")

### Convenios y proveedores

Busqué qué convenios atendía DeltaPay y si compartían proveedor. Como el CSV de pagos no trae proveedor, este cruce justifica la atribución por convenio que uso para calcular el ratio.

In [ ]:
consultar("""
SELECT
    convenio,
    proveedor,
    COUNT(*) AS cantidad_consultas
FROM consultas
GROUP BY convenio, proveedor
ORDER BY convenio, proveedor
""")

## Exploraciones previas

Estas pruebas muestran cómo fui acotando las alternativas antes de comparar la regla final.

# Paso 2 · Validar los datos antes del ratio semanal

## Origen y resultado

Cada fila del resultado cuenta intentos de consulta. Una cuenta puede devolver `deuda_encontrada` varias veces.

In [ ]:
consultar("""
SELECT origen, resultado, COUNT(*) AS intentos
FROM consultas
GROUP BY origen, resultado
ORDER BY origen, intentos DESC
""")

## Fechas cubiertas

Los timestamps vienen en formato ISO. Tomamos los primeros diez caracteres para conservar el día local registrado en el CSV.

In [ ]:
consultar("""
SELECT 'cuentas' AS tabla, MIN(fecha_alta) AS desde, MAX(fecha_alta) AS hasta
FROM cuentas
UNION ALL
SELECT 'consultas', MIN(SUBSTR("timestamp", 1, 10)), MAX(SUBSTR("timestamp", 1, 10))
FROM consultas
UNION ALL
SELECT 'pagos', MIN(SUBSTR(fecha_pago, 1, 10)), MAX(SUBSTR(fecha_pago, 1, 10))
FROM pagos
""")

## Campos vacíos según resultado

En consultas sin deuda, es esperable que monto y vencimiento estén vacíos. Revisá en especial la fila de `deuda_encontrada`.

In [ ]:
consultar("""
SELECT
    resultado,
    COUNT(*) AS intentos,
    COUNT(*) FILTER (WHERE NULLIF(TRIM(monto), '') IS NULL) AS sin_monto,
    COUNT(*) FILTER (WHERE NULLIF(TRIM(fecha_vencimiento), '') IS NULL) AS sin_vencimiento
FROM consultas
GROUP BY resultado
ORDER BY resultado
""")

## Paso 3 · Ratio semanal de DeltaPay

Dividimos consultas por pagos de los convenios de DeltaPay. Las semanas van de lunes a domingo, usando la fecha local del CSV. Marcamos los extremos parciales y dejamos indefinido el ratio cuando hay cero pagos. El exceso supone que se mantienen los pagos observados.

In [ ]:
consultar("""
WITH convenios_delta AS (
    SELECT DISTINCT convenio FROM consultas WHERE proveedor = 'deltapay'
),
consultas_semanales AS (
    SELECT date_trunc('week', CAST(substr("timestamp", 1, 10) AS DATE)) AS semana,
           COUNT(*) AS consultas
    FROM consultas
    WHERE proveedor = 'deltapay'
    GROUP BY 1
),
pagos_semanales AS (
    SELECT date_trunc('week', CAST(substr(fecha_pago, 1, 10) AS DATE)) AS semana,
           COUNT(*) AS pagos
    FROM pagos
    WHERE convenio IN (SELECT convenio FROM convenios_delta)
    GROUP BY 1
)
SELECT semana, COALESCE(consultas, 0) AS consultas, COALESCE(pagos, 0) AS pagos,
       ROUND(COALESCE(consultas, 0) * 1.0 / NULLIF(pagos, 0), 2) AS consultas_por_pago,
       CASE WHEN pagos > 0 THEN GREATEST(COALESCE(consultas, 0) - 6 * pagos, 0) END AS exceso,
       semana < DATE '2026-08-01' OR semana + INTERVAL '6 days' > DATE '2026-10-31' AS semana_parcial
FROM consultas_semanales
FULL OUTER JOIN pagos_semanales USING (semana)
ORDER BY semana
""")

## Hipótesis · Guardar una deuda ya detectada

Usamos cuenta, convenio, monto y vencimiento como aproximación a una factura. Contamos las detecciones posteriores de esa combinación si no hubo un pago registrado entre ambas. Es un conteo histórico de repeticiones, no una estimación directa de ahorro. Una regla operativa necesitaría invalidar al pagar y refrescar el dato cuando pueda cambiar.

In [ ]:
consultar("""
WITH detecciones AS (
    SELECT consulta_id, account_id, convenio, monto, fecha_vencimiento, "timestamp",
           LAG("timestamp") OVER (
               PARTITION BY account_id, convenio, monto, fecha_vencimiento
               ORDER BY "timestamp", consulta_id
           ) AS deteccion_anterior
    FROM consultas
    WHERE proveedor = 'deltapay' AND resultado = 'deuda_encontrada'
      AND monto IS NOT NULL AND fecha_vencimiento IS NOT NULL
)
SELECT convenio,
       COUNT(*) AS consultas_con_deuda,
       COUNT(*) FILTER (
           WHERE deteccion_anterior IS NOT NULL
             AND NOT EXISTS (
                 SELECT 1 FROM pagos p
                 WHERE p.account_id = d.account_id AND p.convenio = d.convenio
                   AND p.monto = d.monto AND p.fecha_vencimiento = d.fecha_vencimiento
                   AND p.fecha_pago >= d.deteccion_anterior
                   AND p.fecha_pago <= d."timestamp"
             )
       ) AS detecciones_repetidas_sin_pago_intermedio
FROM detecciones d
GROUP BY convenio
ORDER BY detecciones_repetidas_sin_pago_intermedio DESC
""")

## Paso 4 · Simular la reutilización de una deuda

Probamos una vigencia de 6, 12 y 24 horas para la última deuda detectada por cuenta y convenio. Volvemos a consultar si la vigencia terminó, si llegó el día del vencimiento o si se registró un pago posterior a la detección para esa cuenta y convenio. Cualquier pago invalida la deuda guardada para esa cuenta. Las consultas omitidas no renuevan la vigencia.

La decisión solo usa eventos conocidos hasta la hora de la consulta. Usamos la respuesta histórica de una consulta omitida para medir qué información se habría perdido. Un cambio de monto o vencimiento cuenta como deuda distinta. El ratio estimado conserva los pagos históricos y no modela cómo podrían cambiar si Tapi detectara una deuda más tarde. El resumen usa las 12 semanas completas.

In [ ]:
import pandas as pd

consultas_delta = consultar("""
    SELECT account_id, convenio, "timestamp", origen, resultado, monto, fecha_vencimiento
    FROM consultas
    WHERE proveedor = 'deltapay'
    ORDER BY "timestamp", consulta_id
""")
pagos_delta = consultar("""
    SELECT account_id, convenio, fecha_pago
    FROM pagos
    WHERE convenio IN (
        SELECT DISTINCT convenio FROM consultas WHERE proveedor = 'deltapay'
    )
""")

# Los CSV usan un único desplazamiento horario (-06:00).
from datetime import date, datetime

consultas_delta["timestamp"] = consultas_delta["timestamp"].map(datetime.fromisoformat)
consultas_delta["fecha_vencimiento"] = consultas_delta["fecha_vencimiento"].map(
    lambda valor: date.fromisoformat(valor) if valor is not None else None
)
pagos_delta["fecha_pago"] = pagos_delta["fecha_pago"].map(datetime.fromisoformat)
eventos = list(consultas_delta.itertuples(index=False))
pagos_observados = list(pagos_delta.itertuples(index=False))


In [ ]:
from bisect import bisect_right
from collections import Counter, defaultdict
from datetime import timedelta

def simular_cache(consultas, pagos, horas, auditoria=None, intervalo_programado=None):
    pagos_por_cuenta = defaultdict(list)
    for pago in pagos:
        pagos_por_cuenta[pago.account_id, pago.convenio].append(pago.fecha_pago)
    for fechas in pagos_por_cuenta.values():
        fechas.sort()

    if isinstance(intervalo_programado, timedelta):
        intervalo_programado = dict.fromkeys(('CONV-TV', 'CONV-AGUA'), intervalo_programado)
    vigencia = timedelta(hours=horas)
    cache = {}
    ultima_programada = {}
    semanas = defaultdict(Counter)

    for consulta in consultas:
        clave = (consulta.account_id, consulta.convenio)
        momento = consulta.timestamp
        dia = momento.date()
        semana = dia - timedelta(days=dia.weekday())
        metricas = semanas[semana]
        metricas["consultas"] += 1

        if auditoria is not None and intervalo_programado is not None:
            if consulta.resultado == "deuda_encontrada":
                factura = (*clave, consulta.fecha_vencimiento)
                auditoria["primera_observada"].setdefault(factura, momento)

        anterior = cache.get(clave)
        if anterior is not None:
            detectada, monto, vencimiento = anterior
            fechas_pago = pagos_por_cuenta[clave]
            siguiente_pago = bisect_right(fechas_pago, detectada)
            hubo_pago = (
                siguiente_pago < len(fechas_pago)
                and fechas_pago[siguiente_pago] <= momento
            )
            if momento - detectada < vigencia and dia < vencimiento and not hubo_pago:
                metricas["omitidas"] += 1
                if consulta.resultado == "deuda_encontrada":
                    if consulta.monto == monto and consulta.fecha_vencimiento == vencimiento:
                        metricas["misma_deuda"] += 1
                    else:
                        metricas["deuda_distinta"] += 1
                        if auditoria is not None:
                            auditoria["cambios_omitidos"].append(consulta)
                        if consulta.fecha_vencimiento is not None and consulta.fecha_vencimiento <= dia:
                            metricas["distinta_vence_hoy_o_antes"] += 1
                else:
                    metricas[consulta.resultado] += 1
                continue

        if (intervalo_programado is not None and consulta.origen == "programada"
            and consulta.convenio in {"CONV-TV", "CONV-AGUA"}):
            ultima = ultima_programada.get(clave)
            if ultima is not None and momento - ultima < intervalo_programado.get(consulta.convenio, timedelta(0)):
                metricas["omitidas"] += 1
                continue
            ultima_programada[clave] = momento

        if auditoria is not None and consulta.resultado == "deuda_encontrada":
            if intervalo_programado is not None:
                factura = (*clave, consulta.fecha_vencimiento)
                auditoria["primera_con_regla"].setdefault(factura, momento)
            auditoria["detecciones_ejecutadas"].append(consulta)
        cache.pop(clave, None)
        if (consulta.resultado == "deuda_encontrada"
            and consulta.monto is not None and consulta.fecha_vencimiento is not None):
            cache[clave] = (momento, consulta.monto, consulta.fecha_vencimiento)

    for pago in pagos:
        dia = pago.fecha_pago.date()
        semanas[dia - timedelta(days=dia.weekday())]["pagos"] += 1

    return semanas


In [ ]:
filas = []
for horas in (6, 12, 24):
    for semana, valores in sorted(simular_cache(eventos, pagos_observados, horas).items()):
        pagos_semana = valores["pagos"]
        consultas_estimadas = valores["consultas"] - valores["omitidas"]
        filas.append({
            "horas": horas,
            "semana": semana,
            "parcial": semana < date(2026, 8, 1) or semana + timedelta(days=6) > date(2026, 10, 31),
            "consultas": valores["consultas"],
            "pagos": pagos_semana,
            "omitidas": valores["omitidas"],
            "ratio_observado": valores["consultas"] / pagos_semana if pagos_semana else None,
            "ratio_estimado": consultas_estimadas / pagos_semana if pagos_semana else None,
            **{clave: valores[clave] for clave in (
                "misma_deuda", "deuda_distinta", "distinta_vence_hoy_o_antes",
                "sin_deuda", "timeout", "error"
            )},
        })

detalle_cache = pd.DataFrame(filas)
completas = detalle_cache.loc[~detalle_cache["parcial"]]
resumen_cache = completas.groupby("horas").agg(
    consultas=("consultas", "sum"),
    omitidas=("omitidas", "sum"),
    misma_deuda=("misma_deuda", "sum"),
    deuda_distinta=("deuda_distinta", "sum"),
    distinta_vence_hoy_o_antes=("distinta_vence_hoy_o_antes", "sum"),
    sin_deuda=("sin_deuda", "sum"),
    timeout=("timeout", "sum"),
    error=("error", "sum"),
    semanas_dentro_del_limite=("ratio_estimado", lambda ratios: (ratios <= 6).sum()),
    menor_ratio_estimado=("ratio_estimado", "min"),
)
resumen_cache["reduccion_pct"] = 100 * resumen_cache["omitidas"] / resumen_cache["consultas"]
display(resumen_cache.round(2))
display(detalle_cache.round(2))


## Paso 5 · Revisar el riesgo de guardar la deuda 12 horas

Por cada respuesta omitida con otro monto o vencimiento, buscamos cuándo una consulta que sí se habría ejecutado devolvió esa misma combinación de cuenta, convenio, monto y vencimiento. Medimos la demora desde la consulta omitida y el margen en días calendario. Una detección el día del vencimiento requiere conocer la hora de corte. Si no volvemos a ver esa combinación en el histórico, la demora queda indeterminada. Cada fila representa una consulta omitida y la misma combinación puede aparecer más de una vez.

In [ ]:
auditoria = {"cambios_omitidos": [], "detecciones_ejecutadas": []}
simular_cache(eventos, pagos_observados, 12, auditoria=auditoria)

def clave_deuda(consulta):
    return (consulta.account_id, consulta.convenio, consulta.monto, consulta.fecha_vencimiento)

detecciones_ejecutadas = defaultdict(list)
for consulta in auditoria["detecciones_ejecutadas"]:
    detecciones_ejecutadas[clave_deuda(consulta)].append(consulta.timestamp)

filas_riesgo = []
for consulta in auditoria["cambios_omitidos"]:
    # Solo reportamos cambios omitidos durante semanas completas.
    if not date(2026, 8, 3) <= consulta.timestamp.date() <= date(2026, 10, 25):
        continue
    fechas = detecciones_ejecutadas[clave_deuda(consulta)]
    indice = bisect_right(fechas, consulta.timestamp)
    siguiente = fechas[indice] if indice < len(fechas) else None
    margen = (
        (consulta.fecha_vencimiento - siguiente.date()).days
        if siguiente is not None and consulta.fecha_vencimiento is not None else None
    )
    filas_riesgo.append({
        "cuenta": consulta.account_id,
        "convenio": consulta.convenio,
        "monto": consulta.monto,
        "vencimiento": consulta.fecha_vencimiento,
        "deteccion_omitida": consulta.timestamp,
        "siguiente_deteccion": siguiente,
        "demora_horas": (siguiente - consulta.timestamp).total_seconds() / 3600 if siguiente else None,
        "dias_hasta_vencimiento": margen,
        "estado": (
            "Sin detección posterior observada" if siguiente is None else
            "Vencimiento desconocido" if margen is None else
            "Después del vencimiento" if margen < 0 else
            "Día del vencimiento" if margen == 0 else "Antes del vencimiento"
        ),
    })

riesgo_cache = pd.DataFrame(filas_riesgo)
display(riesgo_cache.round(2))


## Paso 6 · Frecuencia observada por convenio

Medimos el intervalo entre consultas programadas consecutivas para la misma cuenta y convenio. El promedio diario y el porcentaje de días con varias consultas usan solo días en los que la cuenta tuvo al menos una consulta programada. El ratio del período sirve para comparar convenios. El límite de 6 se evalúa por semana para DeltaPay en conjunto.

Agua y TV concentran el 75,9 % de las consultas de DeltaPay. TV también concentra el 75,1 % de sus reintentos. Gas tiene el mayor ratio del período y 3.620 consultas programadas con resultado `sin_deuda` sobre 4.542 intentos totales. Luz registra una programada por día con actividad y un ratio de 1,94. Estas diferencias orientan qué convenios investigar antes de cambiar el calendario.

In [ ]:
consultar("""
WITH programadas AS (
    SELECT convenio, account_id, SUBSTR("timestamp", 1, 10) AS dia,
           CAST(SUBSTR("timestamp", 1, 19) AS TIMESTAMP) AS momento
    FROM consultas
    WHERE proveedor = 'deltapay' AND origen = 'programada'
),
por_dia AS (
    SELECT convenio, account_id, dia, COUNT(*) AS consultas
    FROM programadas
    GROUP BY 1, 2, 3
),
intervalos AS (
    SELECT convenio,
           momento - LAG(momento) OVER (
               PARTITION BY convenio, account_id ORDER BY momento
           ) AS intervalo
    FROM programadas
),
frecuencia AS (
    SELECT convenio, COUNT(*) AS cuenta_dias_con_consulta,
           AVG(consultas) AS programadas_por_cuenta_dia,
           COUNT(*) FILTER (WHERE consultas > 1) AS cuenta_dias_con_varias
    FROM por_dia GROUP BY convenio
),
medianas AS (
    SELECT convenio, MEDIAN(EPOCH(intervalo) / 3600) AS horas_entre_programadas
    FROM intervalos GROUP BY convenio
),
volumen AS (
    SELECT convenio, COUNT(*) AS consultas,
           COUNT(*) FILTER (WHERE origen = 'programada' AND resultado = 'sin_deuda') AS programadas_sin_deuda,
           COUNT(*) FILTER (WHERE origen = 'reintento') AS reintentos
    FROM consultas WHERE proveedor = 'deltapay' GROUP BY convenio
),
pagos_por_convenio AS (
    SELECT convenio, COUNT(*) AS pagos FROM pagos GROUP BY convenio
)
SELECT convenio, v.consultas, COALESCE(p.pagos, 0) AS pagos,
       ROUND(v.consultas * 1.0 / NULLIF(p.pagos, 0), 2) AS ratio_periodo,
       v.programadas_sin_deuda, v.reintentos,
       ROUND(f.programadas_por_cuenta_dia, 2) AS programadas_por_cuenta_dia,
       ROUND(100.0 * f.cuenta_dias_con_varias / f.cuenta_dias_con_consulta, 1) AS pct_cuenta_dias_con_varias,
       ROUND(m.horas_entre_programadas, 2) AS mediana_horas_entre_programadas
FROM volumen v
LEFT JOIN pagos_por_convenio p USING (convenio)
LEFT JOIN frecuencia f USING (convenio)
LEFT JOIN medianas m USING (convenio)
ORDER BY v.consultas DESC
""")

## Paso 7 · Espaciar consultas programadas en TV y Agua

Simulamos intervalos mínimos de 12 y 24 horas por cuenta y convenio entre consultas programadas que sí se ejecutan. Conservamos los reintentos y pagos observados. Algunos reintentos podrían depender de una consulta omitida, por eso la detección estimada puede ser optimista. Identificamos una deuda por cuenta, convenio y vencimiento como aproximación para seguir su primera detección. Los resultados corresponden a semanas completas.

In [ ]:
from statistics import median

def medir_frecuencia(horas, horas_agua=None, horas_agua_despues=None):
    intervalos = {
        "CONV-TV": timedelta(hours=horas),
        "CONV-AGUA": timedelta(hours=horas if horas_agua is None else horas_agua),
    }
    objetivos = {"CONV-TV", "CONV-AGUA"}
    ultima_programada = {}
    primera_observada = {}
    primera_con_regla = {}
    resumen = defaultdict(Counter)

    def lunes(momento):
        dia = momento.date()
        return dia - timedelta(days=dia.weekday())

    for consulta in eventos:
        momento = consulta.timestamp
        cuenta = (consulta.account_id, consulta.convenio)
        semana = lunes(momento)
        resumen[semana]["consultas"] += 1
        omitir = False
        if consulta.convenio in objetivos and consulta.origen == "programada":
            anterior = ultima_programada.get(cuenta)
            minimo = intervalos[consulta.convenio]
            if consulta.convenio == "CONV-AGUA" and horas_agua_despues is not None and momento.day > 10:
                minimo = timedelta(hours=horas_agua_despues)
            omitir = anterior is not None and momento - anterior < minimo
            if not omitir:
                ultima_programada[cuenta] = momento

        if omitir:
            resumen[semana]["omitidas"] += 1

        if consulta.resultado == "deuda_encontrada" and consulta.fecha_vencimiento is not None:
            clave = (*cuenta, consulta.fecha_vencimiento)
            primera_observada.setdefault(clave, momento)
            if not omitir:
                primera_con_regla.setdefault(clave, momento)

    for pago in pagos_observados:
        resumen[lunes(pago.fecha_pago)]["pagos"] += 1

    semanas_completas = [
        v for semana, v in sorted(resumen.items())
        if date(2026, 8, 3) <= semana <= date(2026, 10, 19)
    ]
    demoras = []
    riesgos = Counter()
    for clave, original in primera_observada.items():
        if clave[1] not in objetivos or not date(2026, 8, 3) <= original.date() <= date(2026, 10, 25):
            continue
        siguiente = primera_con_regla.get(clave)
        if siguiente is None:
            riesgos["sin_deteccion"] += 1
        elif siguiente > original:
            demoras.append((siguiente - original).total_seconds() / 3600)
            if siguiente.date() >= clave[-1] and original.date() < clave[-1]:
                riesgos["pierde_margen_previo"] += 1

    return {
        "horas_tv": horas,
        "horas_agua": horas if horas_agua is None else horas_agua,
        "horas_agua_despues": horas_agua_despues,
        "consultas_omitidas": sum(v["omitidas"] for v in semanas_completas),
        "semanas_en_limite": sum(
            v["pagos"] > 0 and v["consultas"] - v["omitidas"] <= 6 * v["pagos"]
            for v in semanas_completas
        ),
        "mejor_ratio": min((v["consultas"] - v["omitidas"]) / v["pagos"] for v in semanas_completas if v["pagos"]),
        "detecciones_demoradas": len(demoras),
        "mediana_demora_horas": median(demoras) if demoras else None,
        "sin_deteccion_observada": riesgos["sin_deteccion"],
        "pasa_al_dia_de_vencimiento_o_despues": riesgos["pierde_margen_previo"],
    }


display(pd.DataFrame([medir_frecuencia(h) for h in (12, 24)]).round(2))

## Paso 8 · Combinar las reglas de 12 horas

Simulamos ambos cambios sobre el mismo historial. Primero se evalúa si se puede reutilizar la deuda conocida. Para las consultas programadas de TV y Agua que siguen pendientes, exigimos un intervalo de 12 horas desde la última programada ejecutada. Una consulta omitida no actualiza el estado. Se conservan los pagos históricos y los reintentos siguen disponibles para la regla de reutilización aunque su consulta de origen se haya omitido. Identificamos la primera detección por cuenta, convenio y vencimiento, porque no hay ID de factura.

In [ ]:
auditoria_combinada = {
    "cambios_omitidos": [], "detecciones_ejecutadas": [],
    "primera_observada": {}, "primera_con_regla": {},
}
semanas_combinadas = simular_cache(
    eventos, pagos_observados, 12, auditoria=auditoria_combinada,
    intervalo_programado=timedelta(hours=12),
)
filas_combinadas = []
for semana, datos in sorted(semanas_combinadas.items()):
    if not date(2026, 8, 3) <= semana <= date(2026, 10, 19):
        continue
    consultas_restantes = datos["consultas"] - datos["omitidas"]
    pagos = datos["pagos"]
    filas_combinadas.append({
        "semana": semana, "consultas": datos["consultas"], "pagos": pagos,
        "omitidas": datos["omitidas"],
        "ratio_observado": datos["consultas"] / pagos if pagos else None,
        "ratio_combinado": consultas_restantes / pagos if pagos else None,
    })

riesgo_combinado = Counter()
for factura, original in auditoria_combinada["primera_observada"].items():
    vencimiento = factura[-1]
    if vencimiento is None or not date(2026, 8, 3) <= original.date() <= date(2026, 10, 25):
        continue
    deteccion = auditoria_combinada["primera_con_regla"].get(factura)
    if deteccion is None:
        riesgo_combinado["sin_deteccion_posterior"] += 1
    elif deteccion > original:
        riesgo_combinado["primeras_detecciones_demoradas"] += 1
        if original.date() < vencimiento <= deteccion.date():
            riesgo_combinado["pasan_al_vencimiento_o_despues"] += 1

detalle_combinado = pd.DataFrame(filas_combinadas)
display(detalle_combinado.round(2))
print("Consultas omitidas:", detalle_combinado["omitidas"].sum())
print("Semanas dentro del límite:", detalle_combinado["ratio_combinado"].le(6).sum())
for indicador in ("primeras_detecciones_demoradas", "sin_deteccion_posterior", "pasan_al_vencimiento_o_despues"):
    print(indicador, riesgo_combinado[indicador])


## Paso 9 · Probar un intervalo propio para Agua

Fijamos TV en 12 horas y comparamos 12, 18 y 24 horas para Agua, sin aplicar la regla de deuda guardada. Un intervalo mayor puede conservar consultas en otros horarios, por eso la cantidad de primeras detecciones demoradas no necesariamente crece de forma lineal. Se mantienen los pagos y reintentos históricos.

In [ ]:
display(pd.DataFrame([medir_frecuencia(12, horas) for horas in (12, 18, 24)]).round(2))

## Paso 10 · Combinar TV a 12 h, Agua a 18 h y deuda guardada a 12 h

La brecha semanal muestra cuántas consultas adicionales habría que evitar para llegar a 6 con los pagos observados. Se mantienen los reintentos históricos disponibles y se aproxima la factura por cuenta, convenio y vencimiento.

In [ ]:
auditoria_combinada = {
    "cambios_omitidos": [], "detecciones_ejecutadas": [],
    "primera_observada": {}, "primera_con_regla": {},
}
semanas_combinadas = simular_cache(
    eventos, pagos_observados, 12, auditoria=auditoria_combinada,
    intervalo_programado={'CONV-TV': timedelta(hours=12), 'CONV-AGUA': timedelta(hours=18)},
)
filas_combinadas = []
for semana, datos in sorted(semanas_combinadas.items()):
    if not date(2026, 8, 3) <= semana <= date(2026, 10, 19):
        continue
    consultas_restantes = datos["consultas"] - datos["omitidas"]
    pagos = datos["pagos"]
    filas_combinadas.append({
        "semana": semana, "consultas": datos["consultas"], "pagos": pagos,
        "omitidas": datos["omitidas"],
        "consultas_a_reducir": max(consultas_restantes - 6 * pagos, 0),
        "ratio_observado": datos["consultas"] / pagos if pagos else None,
        "ratio_combinado": consultas_restantes / pagos if pagos else None,
    })

riesgo_combinado = Counter()
for factura, original in auditoria_combinada["primera_observada"].items():
    vencimiento = factura[-1]
    if vencimiento is None or not date(2026, 8, 3) <= original.date() <= date(2026, 10, 25):
        continue
    deteccion = auditoria_combinada["primera_con_regla"].get(factura)
    if deteccion is None:
        riesgo_combinado["sin_deteccion_posterior"] += 1
    elif deteccion > original:
        riesgo_combinado["primeras_detecciones_demoradas"] += 1
        if original.date() < vencimiento <= deteccion.date():
            riesgo_combinado["pasan_al_vencimiento_o_despues"] += 1

detalle_combinado = pd.DataFrame(filas_combinadas)
display(detalle_combinado.round(2))
print("Consultas omitidas:", detalle_combinado["omitidas"].sum())
print("Semanas dentro del límite:", detalle_combinado["ratio_combinado"].le(6).sum())
for indicador in ("primeras_detecciones_demoradas", "sin_deteccion_posterior", "pasan_al_vencimiento_o_despues"):
    print(indicador, riesgo_combinado[indicador])


## Paso 11 · Calendario de pagos y vencimientos

Cruzamos la semana de pago con la semana de vencimiento de las facturas pagadas. El corte mensual por días 1 a 7 permite comparar el comienzo de cada mes con la misma ventana. El CSV de pagos solo muestra vencimientos de facturas pagadas.

In [ ]:
consultar("""
SELECT convenio,
       date_trunc('week', CAST(substr(fecha_pago,1,10) AS DATE)) AS semana_pago,
       date_trunc('week', CAST(fecha_vencimiento AS DATE)) AS semana_vencimiento,
       COUNT(*) AS pagos
FROM pagos
WHERE convenio IN (SELECT DISTINCT convenio FROM consultas WHERE proveedor = 'deltapay')
GROUP BY 1, 2, 3
ORDER BY 2, 1, 3
""")

Busqué si el bajo número de pagos al comienzo del mes se repetía, contando los pagos de los primeros siete días y comparándolos con el total mensual.

In [ ]:
consultar("""
SELECT substr(fecha_pago, 1, 7) AS mes,
       COUNT(*) AS pagos,
       COUNT(*) FILTER (WHERE substr(fecha_pago, 9, 2) <= '07') AS pagos_dias_1_a_7,
       ROUND(100.0 * pagos_dias_1_a_7 / COUNT(*), 1) AS pct_dias_1_a_7
FROM pagos
WHERE convenio IN (SELECT DISTINCT convenio FROM consultas WHERE proveedor = 'deltapay')
GROUP BY 1
ORDER BY 1
""")

## Paso 12 · Picos de pago y detección por convenio

Agrupamos por día del mes y mostramos los tres mayores conteos, incluidos los empates. La primera detección se aproxima por cuenta, convenio y vencimiento. El día en que aparece en los datos depende del calendario histórico de consultas. Usamos los picos para definir ventanas candidatas y evaluar su impacto en una simulación.

In [ ]:
consultar("""
WITH convenios_delta AS (
    SELECT DISTINCT convenio FROM consultas WHERE proveedor = 'deltapay'
),
primeras_detecciones AS (
    SELECT convenio, account_id, fecha_vencimiento,
           MIN(SUBSTR("timestamp", 1, 10)) AS fecha
    FROM consultas
    WHERE proveedor = 'deltapay' AND resultado = 'deuda_encontrada'
      AND fecha_vencimiento IS NOT NULL
    GROUP BY 1, 2, 3
),
eventos_por_dia AS (
    SELECT convenio, SUBSTR(fecha_pago, 1, 10) AS fecha, 'Pago' AS evento
    FROM pagos WHERE convenio IN (SELECT convenio FROM convenios_delta)
    UNION ALL
    SELECT convenio, fecha, 'Primera detección' FROM primeras_detecciones
),
conteos AS (
    SELECT convenio, evento, SUBSTR(fecha, 9, 2) AS dia_del_mes,
           COUNT(*) AS cantidad
    FROM eventos_por_dia
    GROUP BY 1, 2, 3
)
SELECT *, DENSE_RANK() OVER (
    PARTITION BY convenio, evento ORDER BY cantidad DESC
) AS puesto
FROM conteos
QUALIFY puesto <= 3
ORDER BY convenio, evento, puesto, dia_del_mes
""")

## Paso 13 · Frecuencia de Agua por día del mes

Comparamos 18 h fijas con 18 h del día 1 al 10 y 24 h desde el día 11, usando el día local de cada consulta programada. TV queda a 12 h en ambos escenarios. Conservamos los pagos y reintentos observados. La primera detección se aproxima por cuenta, convenio y vencimiento.

In [ ]:
display(pd.DataFrame([medir_frecuencia(12, 18), medir_frecuencia(12, 18, 24)],
                     index=["18 h fijas", "18 h hasta el día 10, luego 24 h"]).round(2))

## Paso 14 · Consultar según el estado de deuda

Probamos intervalos comunes de 24, 48 y 72 horas mientras no hay deuda conocida. Con deuda, suspendemos las consultas hasta el pago o hasta un día antes del vencimiento. El respaldo usa las consultas del histórico que quedan disponibles. Sumamos una revalidación hipotética por cada pago observado, sin asignarle una respuesta. Los pagos quedan fijos. Identificamos deudas por cuenta, convenio y vencimiento, y medimos si reaparecen entre las consultas retenidas. Esta primera prueba usa intervalos comunes como referencia para diseñar reglas por convenio.

In [ ]:
import csv
from collections import Counter
from datetime import datetime, timedelta, date
from pathlib import Path

def simular_estado(consultas, pagos, intervalo):
    eventos = [(datetime.fromisoformat(r['timestamp']), 1, r['consulta_id'], r) for r in consultas]
    eventos += [(datetime.fromisoformat(r['fecha_pago']), 0, r['pago_id'], r) for r in pagos]
    estado, primera, retenida = {}, {}, {}
    semanal = {}
    for t, tipo, _, r in sorted(eventos):
        dia = t.date()
        lunes = dia - timedelta(days=dia.weekday())
        m = semanal.setdefault(lunes, Counter())
        k = r['account_id'], r['convenio']
        if tipo == 0:
            m['pagos'] += 1
            m['revalidaciones'] += 1
            estado.pop(k, None)
            continue
        m['observadas'] += 1
        deuda = r['resultado'] == 'deuda_encontrada' and r['fecha_vencimiento']
        if deuda:
            clave = (*k, date.fromisoformat(r['fecha_vencimiento']))
            primera.setdefault(clave, t)
        ultima, vence = estado.get(k, (None, None))
        if ultima is not None and (t-ultima < timedelta(hours=intervalo) or (vence and dia < vence-timedelta(days=1))):
            # El respaldo prevalece cuando falta un día para el vencimiento conocido.
            if not vence or dia < vence-timedelta(days=1):
                continue
        m['ejecutadas'] += 1
        if deuda:
            retenida.setdefault(clave, t)
        nuevo_vence = clave[-1] if deuda else vence if r['resultado'] in ('timeout','error') else None
        estado[k] = (t, nuevo_vence)
    riesgo = Counter()
    for clave, t in primera.items():
        if not date(2026,8,3) <= t.date() <= date(2026,10,25):
            continue
        posterior = retenida.get(clave)
        if posterior is None:
            riesgo['sin_deteccion'] += 1
        elif posterior > t:
            riesgo['demoradas'] += 1
            if t.date() < clave[-1] <= posterior.date():
                riesgo['pierden_margen'] += 1
    return semanal, riesgo


def leer_estado(nombre):
    with (Path("data") / f"B_agenda_{nombre}.csv").open() as archivo:
        return list(csv.DictReader(archivo))

consultas_estado = [r for r in leer_estado("consultas") if r["proveedor"] == "deltapay"]
convenios_estado = {r["convenio"] for r in consultas_estado}
pagos_estado = [r for r in leer_estado("pagos") if r["convenio"] in convenios_estado]
filas_estado = []
for horas in (24, 48, 72):
    semanas, riesgo = simular_estado(consultas_estado, pagos_estado, horas)
    for semana, datos in sorted(semanas.items()):
        if date(2026, 8, 3) <= semana <= date(2026, 10, 19):
            consultas = datos["ejecutadas"] + datos["revalidaciones"]
            filas_estado.append({"horas": horas, "semana": semana, "consultas": consultas,
                                 "pagos": datos["pagos"], "ratio": consultas / datos["pagos"] if datos["pagos"] else None})
    print(horas, "horas:", dict(riesgo))
display(pd.DataFrame(filas_estado).round(2))


## 1. ¿En qué semanas se supera el límite?

Conté todos los intentos de DeltaPay y los pagos atribuidos a sus convenios, agrupados por semana. Mostré ambos conteos para distinguir si el ratio alto venía de muchas consultas, de pocos pagos o de ambos. Usé las 12 semanas completas del 3 de agosto al 25 de octubre de 2026 y la fecha local del CSV.

In [1]:
consultar("""
WITH q AS (
    SELECT CAST(date_trunc('week', CAST(left(timestamp, 10) AS DATE)) AS DATE) AS semana,
           count(*) AS consultas
    FROM consultas WHERE proveedor = 'deltapay'
    GROUP BY 1
), p AS (
    SELECT CAST(date_trunc('week', CAST(left(fecha_pago, 10) AS DATE)) AS DATE) AS semana,
           count(*) AS pagos
    FROM pagos
    WHERE convenio IN (SELECT DISTINCT convenio FROM consultas WHERE proveedor = 'deltapay')
    GROUP BY 1
)
SELECT semana, consultas, coalesce(pagos, 0) AS pagos,
       round(consultas * 1.0 / nullif(pagos, 0), 2) AS ratio
FROM q LEFT JOIN p USING (semana)
WHERE semana BETWEEN DATE '2026-08-03' AND DATE '2026-10-19'
ORDER BY semana
""")

,semana,consultas,pagos,ratio
0,2026-08-03,3046,8,380.75
1,2026-08-10,3641,131,27.79
2,2026-08-17,2633,169,15.58
3,2026-08-24,1824,139,13.12
4,2026-08-31,2905,40,72.63
5,2026-09-07,3405,76,44.80
6,2026-09-14,3415,172,19.85
7,2026-09-21,1944,160,12.15
8,2026-09-28,2813,76,37.01
9,2026-10-05,3290,42,78.33


### ¿En qué convenios conviene enfocarse?

Calculé el peso de cada convenio en el volumen de DeltaPay. TV y Agua sumaron 75,9 %, así que concentré las primeras pruebas de frecuencia en esos dos.

In [2]:
consultar("""
SELECT convenio, count(*) AS consultas,
       round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS porcentaje
FROM consultas
WHERE proveedor = 'deltapay'
  AND left(timestamp, 10) BETWEEN '2026-08-03' AND '2026-10-25'
GROUP BY convenio ORDER BY consultas DESC
""")

,convenio,consultas,porcentaje
0,CONV-TV,14810,42.80
1,CONV-AGUA,11437,33.05
2,CONV-GAS,4227,12.22
3,CONV-TEL,2644,7.64
4,CONV-LUZ,1484,4.29


## 2. ¿Qué consultas parecen evitables?

Las 12 semanas superaron 6. TV y Agua concentraron el volumen. Después busqué facturas que ya se habían detectado antes de pagar. Vinculé por cuenta, convenio y vencimiento. La primera consulta positiva anterior al pago sirve de punto de partida. Comparo monto de cada reconsulta con el de esa primera respuesta. El vencimiento identifica la factura dentro de cada cuenta y convenio en esta aproximación.

Esta consulta usa todo el archivo de pagos. Al exigir que el monto coincida con el pago, 630 de 1.322 pagos vinculados tienen reconsultas (47,7 %), con una mediana de 12 reconsultas de igual monto entre los casos con repetición. Para medir cambios de importe uso también el vínculo sin ese filtro.


In [3]:
consultar("""
WITH filtros AS (
    SELECT unnest([false, true]) AS exigir_monto_pago
), vinculos AS (
    SELECT exigir_monto_pago, p.pago_id, q.monto,
           row_number() OVER (PARTITION BY exigir_monto_pago, p.pago_id ORDER BY q.timestamp, q.consulta_id) AS orden,
           first_value(q.monto) OVER (PARTITION BY exigir_monto_pago, p.pago_id ORDER BY q.timestamp, q.consulta_id) AS monto_inicial
    FROM pagos p JOIN consultas q
      ON p.account_id = q.account_id AND p.convenio = q.convenio
     AND p.fecha_vencimiento = q.fecha_vencimiento
     AND q.timestamp < p.fecha_pago
    CROSS JOIN filtros
    WHERE (NOT exigir_monto_pago OR TRY_CAST(q.monto AS DECIMAL(18,2)) = TRY_CAST(p.monto AS DECIMAL(18,2)))
      AND q.proveedor = 'deltapay' AND q.resultado = 'deuda_encontrada'
), por_pago AS (
    SELECT exigir_monto_pago, pago_id, count(*) FILTER (WHERE orden > 1) AS reconsultas,
           count(*) FILTER (WHERE orden > 1 AND monto = monto_inicial) AS mismo_monto,
           count(*) FILTER (WHERE orden > 1 AND monto <> monto_inicial) AS cambio_monto
    FROM vinculos GROUP BY exigir_monto_pago, pago_id
)
SELECT exigir_monto_pago, count(*) AS pagos_vinculados,
       count(*) FILTER (WHERE reconsultas > 0) AS pagos_con_reconsulta,
       median(mismo_monto) FILTER (WHERE mismo_monto > 0) AS mediana_reconsultas_mismo_monto,
       count(*) FILTER (WHERE cambio_monto > 0) AS pagos_con_cambio_monto
FROM por_pago GROUP BY exigir_monto_pago ORDER BY exigir_monto_pago
""")

,exigir_monto_pago,pagos_vinculados,pagos_con_reconsulta,mediana_reconsultas_mismo_monto,pagos_con_cambio_monto
0,False,1432,642,12.0,93
1,True,1322,630,12.0,0


## 3. Simular una regla según si Tapi conoce la deuda

La idea fue espaciar la búsqueda y pausar la rutina cuando hay una factura conocida. Mantengo una consulta de respaldo cerca del vencimiento y contabilizo una revalidación hipotética antes de cada pago de los convenios modificados. Los pagos históricos se mantienen fijos. El replay conserva o descarta intentos observados: una factura ausente en las respuestas conservadas podría aparecer con otra programación real. La simulación puede conservar intentos históricos adicionales desde el día anterior al vencimiento.

In [ ]:
import csv
from collections import Counter
from datetime import datetime, timedelta, date
from pathlib import Path

def simular_estado(consultas, pagos, intervalo, ventanas=None, reintento_dias=()):
    eventos = [(datetime.fromisoformat(r['timestamp']), 1, r['consulta_id'], r) for r in consultas]
    eventos += [(datetime.fromisoformat(r['fecha_pago']), 0, r['pago_id'], r) for r in pagos]
    estado, primera, retenida = {}, {}, {}
    guardadas=[]
    pendientes=set()
    semanal = {}
    for t, tipo, _, r in sorted(eventos):
        dia = t.date()
        lunes = dia - timedelta(days=dia.weekday())
        m = semanal.setdefault(lunes, Counter())
        k = r['account_id'], r['convenio']
        if tipo == 0:
            m['pagos'] += 1
            m['revalidaciones'] += 1
            estado.pop(k, None)
            pendientes.discard(k)
            continue
        m['observadas'] += 1
        deuda = r['resultado'] == 'deuda_encontrada' and r['fecha_vencimiento']
        if deuda:
            clave = (*k, date.fromisoformat(r['fecha_vencimiento']))
            primera.setdefault(clave, t)
        ultima, vence = estado.get(k, (None, None))
        horas = intervalo
        ventana = (ventanas or {}).get(k)
        if dia.month == 10 and not vence and ventana and not ventana[0] <= dia.day <= ventana[1]:
            horas = 96
        es_reintento = k in pendientes and ultima is not None and t-ultima < timedelta(hours=horas)
        if not es_reintento and ultima is not None and (t-ultima < timedelta(hours=horas) or (vence and dia < vence-timedelta(days=1))):
            # El respaldo prevalece cuando falta un día para el vencimiento conocido.
            if not vence or dia < vence-timedelta(days=1):
                continue
        m['ejecutadas'] += 1
        guardadas.append(r)
        pendientes.discard(k)
        if not es_reintento and r['resultado'] in ('timeout','error') and vence is None and dia.day in reintento_dias:
            pendientes.add(k)
        if deuda:
            retenida.setdefault(clave, t)
        nuevo_vence = clave[-1] if deuda else vence if r['resultado'] in ('timeout','error') else None
        estado[k] = (ultima if es_reintento else t, nuevo_vence)
    return semanal, primera, retenida, guardadas

### ¿Qué ahorro y qué riesgo trae cada frecuencia?

Comparé el volumen, las semanas dentro del límite y las facturas del histórico que ya no aparecen en las respuestas conservadas. Así puedo ver si ahorrar más consultas cambia el cumplimiento semanal.

In [4]:
def leer(nombre):
    with (Path("data") / nombres[nombre]).open() as archivo:
        return list(csv.DictReader(archivo))

consultas_delta = [r for r in leer("consultas") if r["proveedor"] == "deltapay"]
convenios = sorted({r["convenio"] for r in consultas_delta})
pagos_delta = [r for r in leer("pagos") if r["convenio"] in convenios]

def evaluar(intervalo, luz_historica=True, tv=None, ventanas=None, reintento_dias=()):
    consultas_semana, pagos_semana = Counter(), Counter()
    primeras, detecciones = {}, {}
    for convenio in convenios:
        cc = [r for r in consultas_delta if r["convenio"] == convenio]
        pp = [r for r in pagos_delta if r["convenio"] == convenio]
        horas = tv if convenio == "CONV-TV" and tv is not None else intervalo
        dias = reintento_dias if convenio == "CONV-TV" else ()
        semanas, original, retenida, _ = simular_estado(cc, pp, horas, ventanas, dias)
        sin_cambios = luz_historica and convenio == "CONV-LUZ"
        primeras.update(original)
        detecciones.update(original if sin_cambios else retenida)
        for semana, datos in semanas.items():
            consultas_semana[semana] += datos["observadas"] if sin_cambios else datos["ejecutadas"] + datos["revalidaciones"]
            pagos_semana[semana] += datos["pagos"]
    return consultas_semana, pagos_semana, primeras, detecciones

def resumir(resultado, desde, hasta, desde_facturas, hasta_facturas):
    consultas, pagos, primeras, detecciones = resultado
    semanas = [s for s in sorted(pagos) if desde <= s <= hasta]
    q, p = sum(consultas[s] for s in semanas), sum(pagos[s] for s in semanas)
    ausentes = Counter(k[1] for k,t in primeras.items()
                       if desde_facturas <= t.date() <= hasta_facturas and k not in detecciones)
    return q, round(q/p, 2), sum(0 < pagos[s] and consultas[s] <= 6*pagos[s] for s in semanas), dict(ausentes)

base = evaluar(48)
periodo = (date(2026,8,3), date(2026,10,19), date(2026,8,3), date(2026,10,25))
escenarios = {
    "48 h para todos": evaluar(48, luz_historica=False),
    "48 h salvo Luz": base,
    "72 h salvo Luz": evaluar(72),
    "96 h salvo Luz": evaluar(96),
    "TV 24 h": evaluar(48, tv=24),
    "TV 24 h y reintento cualquier día": evaluar(48, tv=24, reintento_dias=range(1,32)),
    "TV 24 h y reintento en ventanas": evaluar(48, tv=24, reintento_dias={5,6,7,20,21,22,23,24}),
}
import pandas as pd
pd.DataFrame({nombre: resumir(r, *periodo) for nombre,r in escenarios.items()},
             index=["Consultas", "Ratio", "Semanas ≤ 6", "Facturas ausentes por convenio"]).T

,Consultas,Ratio,Semanas ≤ 6,Facturas ausentes por convenio
48 h para todos,6722,5.01,7,"{'CONV-LUZ': 724, 'CONV-TV': 7}"
48 h salvo Luz,6739,5.02,7,{'CONV-TV': 7}
72 h salvo Luz,5720,4.26,7,{'CONV-TV': 15}
96 h salvo Luz,5183,3.86,8,"{'CONV-AGUA': 2, 'CONV-TV': 20}"
TV 24 h,7886,5.87,7,{'CONV-TV': 3}
TV 24 h y reintento cualquier día,8265,6.15,6,{}
TV 24 h y reintento en ventanas,7966,5.93,7,{}


### Qué aprendí de las simulaciones

Con 48 h para todos, 724 de las 731 facturas ausentes eran de Luz. Su programación histórica habitualmente agrupa dos consultas en días consecutivos por cuenta cerca de una vez al mes. Al conservarla, quedan siete casos de TV y un ratio de 5,02. Con 96 h se suma una semana dentro del límite, pero se agregan 15 facturas ausentes. Revisé TV para entender qué pasó antes de decidir si valía la pena sumar una excepción.

### Qué pasó con las facturas de TV que dejaron de aparecer

Partí de las siete facturas de TV ausentes con 48 h. Primero verifiqué cuáles reaparecían al pasar a 24 h y si surgían casos nuevos. Para las tres ausentes con 24 h, reconstruí la secuencia por cuenta y vencimiento: primera detección, consultas conservadas con timeout antes del pago, pago histórico y detección con un reintento. La comparación usa consultas históricas y mantiene fijos los pagos.

In [1]:
consultas_tv = [r for r in consultas_delta if r["convenio"] == "CONV-TV"]
pagos_tv = [r for r in pagos_delta if r["convenio"] == "CONV-TV"]
_, _, _, intentos_tv_24h = simular_estado(consultas_tv, pagos_tv, 24)
detecciones_24h = escenarios["TV 24 h"][3]
detecciones_reintento = escenarios["TV 24 h y reintento en ventanas"][3]
filas_tv = []
for clave, primera in sorted(base[2].items(), key=lambda par: par[1]):
    cuenta, convenio, vencimiento = clave
    if convenio != "CONV-TV" or (clave in base[3] and clave in detecciones_24h) or not periodo[2] <= primera.date() <= periodo[3]:
        continue
    pago = min((datetime.fromisoformat(r["fecha_pago"]) for r in pagos_tv
                if r["account_id"] == cuenta and r["fecha_vencimiento"] == str(vencimiento)
                and datetime.fromisoformat(r["fecha_pago"]) > primera), default=None)
    timeouts = sorted(datetime.fromisoformat(r["timestamp"]) for r in intentos_tv_24h
                      if r["account_id"] == cuenta and r["resultado"] == "timeout"
                      and pago is not None and primera <= datetime.fromisoformat(r["timestamp"]) < pago)
    recuperada = detecciones_reintento.get(clave)
    filas_tv.append((cuenta, primera, vencimiento, pago, base[3].get(clave), detecciones_24h.get(clave),
                     ", ".join(t.strftime("%d/%m %H:%M") for t in timeouts), recuperada))
pd.DataFrame(filas_tv, columns=["Cuenta", "Primera detección", "Vencimiento", "Pago histórico",
                               "Detección con 48 h", "Detección con 24 h", "Timeouts antes del pago", "Con reintento en ventanas"])

Cuenta,Primera detección,Vencimiento,Pago histórico,Detección con 48 h,Detección con 24 h,Timeouts antes del pago,Con reintento en ventanas
acc_00207,2026-08-06 09:00:25-06:00,2026-08-13,2026-08-11 15:38:00-06:00,NaT,NaT,"06/08 09:02, 08/08 09:01, 10/08 09:01",2026-08-07 09:00:00-06:00
acc_00158,2026-08-20 09:05:00-06:00,2026-08-26,2026-08-22 09:14:00-06:00,2026-08-21 09:05:25-06:00,NaT,"21/08 09:01, 22/08 09:02",2026-08-21 09:05:25-06:00
acc_00215,2026-08-24 09:00:25-06:00,2026-09-01,2026-08-27 12:31:00-06:00,NaT,2026-08-24 09:00:25-06:00,,2026-08-24 09:00:25-06:00
acc_00156,2026-09-13 09:02:00-06:00,2026-09-22,2026-09-18 12:33:00-06:00,NaT,2026-09-13 09:02:00-06:00,,2026-09-13 09:02:00-06:00
acc_00213,2026-09-21 09:01:00-06:00,2026-09-28,2026-09-27 19:54:00-06:00,NaT,2026-09-21 09:01:00-06:00,"27/09 09:01, 27/09 09:07",2026-09-21 09:01:00-06:00
acc_00235,2026-10-06 09:00:25-06:00,2026-10-12,2026-10-09 16:18:00-06:00,NaT,2026-10-06 09:00:25-06:00,,2026-10-06 09:00:25-06:00
acc_00135,2026-10-19 09:07:00-06:00,2026-10-25,2026-10-21 10:17:00-06:00,NaT,2026-10-19 09:07:00-06:00,,2026-10-19 09:07:00-06:00
acc_00237,2026-10-23 09:01:00-06:00,2026-10-29,2026-10-25 12:04:00-06:00,NaT,NaT,24/10 09:00,2026-10-24 09:04:50-06:00


Con 24 h reaparecen cinco de las siete facturas que faltaban con 48 h. Dos siguen ausentes y aparece un caso nuevo, así que el total baja de siete a tres. En esos tres casos hubo un timeout en una consulta conservada antes del pago histórico. Con un reintento en las ventanas del 5 al 7 y del 20 al 24, las tres reaparecen antes del pago histórico. Las fechas salieron de esos mismos casos y el escenario sigue en 7 de 12 semanas dentro del límite. Por eso dejé las ventanas fuera de la propuesta principal.

## 4. ¿Se repiten las fechas en que aparecen las facturas?

Usé el día de primera detección como aproximación a cuándo apareció la factura. Para cada convenio calculé la ventana de cinco días con más detecciones y medí la repetición por cuenta en meses consecutivos. Estas ventanas describen el histórico completo. La regla evaluada a continuación aprende sus fechas exclusivamente de agosto y septiembre.

In [5]:
from collections import defaultdict

primeras = base[2]
por_cuenta = defaultdict(list)
for (cuenta, convenio, vencimiento), fecha in primeras.items():
    por_cuenta[(cuenta, convenio)].append(fecha)
filas = []
for convenio in convenios:
    grupos = [sorted(fechas) for (cuenta, cv), fechas in por_cuenta.items() if cv == convenio]
    dias = Counter(t.day for fechas in grupos for t in fechas)
    inicio = max(range(1,28), key=lambda dia: sum(dias[d] for d in range(dia,dia+5)))
    pares = [(a,b) for fechas in grupos for a,b in zip(fechas, fechas[1:])
             if b.year*12+b.month == a.year*12+a.month+1]
    filas.append((convenio, f"{inicio}–{inicio+4}", sum(dias[d] for d in range(inicio,inicio+5)),
                  sum(dias.values()), sum(abs(a.day-b.day)<=2 for a,b in pares), len(pares)))
pd.DataFrame(filas, columns=["Convenio","Ventana","En ventana","Facturas","Repite ±2 días","Pares"])

,Convenio,Ventana,En ventana,Facturas,Repite ±2 días,Pares
0,CONV-AGUA,2–6,87,318,71,190
1,CONV-GAS,2–6,12,37,0,2
2,CONV-LUZ,5–9,202,744,176,458
3,CONV-TEL,12–16,22,63,15,36
4,CONV-TV,18–22,67,285,58,183


## 5. Evaluación en octubre con entrenamiento en agosto y septiembre

Quise saber si podía ahorrar consultas usando el patrón de cada cuenta. Para definir la regla usé únicamente agosto y septiembre. Fuera de Luz, seleccioné cuentas con una primera detección en cada mes y hasta dos días de diferencia entre ambos días del mes. La ventana va desde dos días antes de la fecha más temprana hasta dos después de la más tardía.

Durante octubre mantuve 48 h dentro de la ventana y probé 96 h fuera de ella mientras se busca deuda. Las demás cuentas mantienen su regla base. Comparé el ratio en las tres semanas completas de octubre y evalué la detección sobre las facturas que aparecieron por primera vez durante todo el mes.

### ¿La regla aprendida mejora el resultado en octubre?

Construí las ventanas con los meses de entrenamiento y comparé cada semana con la regla de 48 h. Dejé fijos los pagos y conté aparte las facturas de octubre que no aparecen en el replay.

In [6]:
ventanas_octubre = {}
for clave, fechas in por_cuenta.items():
    if clave[1] == "CONV-LUZ":
        continue
    agosto = [t.day for t in fechas if t.month == 8]
    septiembre = [t.day for t in fechas if t.month == 9]
    if len(agosto) == len(septiembre) == 1 and abs(agosto[0]-septiembre[0]) <= 2:
        dias = agosto + septiembre
        ventanas_octubre[clave] = (min(dias)-2, max(dias)+2)

octubre = evaluar(48, ventanas=ventanas_octubre)
print("Cuentas elegibles:", len(ventanas_octubre))
for nombre, resultado in [("48 h", base), ("Regla por cuenta", octubre)]:
    print(nombre, resumir(resultado, date(2026,10,5), date(2026,10,19), date(2026,10,1), date(2026,10,31)))

filas = [(s, base[1][s], base[0][s], octubre[0][s],
          round(base[0][s]/base[1][s],2), round(octubre[0][s]/base[1][s],2))
         for s in sorted(base[1]) if date(2026,10,5) <= s <= date(2026,10,19)]
pd.DataFrame(filas, columns=["Semana", "Pagos", "Consultas 48 h", "Consultas por cuenta",
                             "Ratio 48 h", "Ratio por cuenta"])

Cuentas elegibles: 65
48 h (1676, 4.51, 2, {'CONV-TV': 3})
Regla por cuenta (1615, 4.34, 2, {'CONV-TV': 6})


,Semana,Pagos,Consultas 48 h,Consultas por cuenta,Ratio 48 h,Ratio por cuenta
0,2026-10-05,42,595,570,14.17,13.57
1,2026-10-12,167,679,653,4.07,3.91
2,2026-10-19,163,402,392,2.47,2.40


## 6. Decisión para el piloto

La regla por cuenta ahorró 61 consultas en las tres semanas completas de octubre. Ambas alternativas cumplieron dos semanas y la regla por cuenta dejó tres facturas adicionales de TV ausentes en las consultas conservadas.

Propongo mantener 48 h para buscar una factura, preservar la programación de Luz, pausar las consultas de rutina cuando Tapi conoce la deuda y confirmar el importe antes de pagar. En la simulación del período, el ratio pasó de 25,76 a 5,02 y cumplió 7 de 12 semanas. Para decidir si ampliar el piloto mediría el ratio por semana, cuánto margen queda al detectar cada factura y cuántos pagos se completan a tiempo. La simulación mantiene fijos los pagos, por eso la puntualidad se valida en el piloto.